# 07 · Auditoria do ensaio Krijnen / CGAL e incorporação ao RDF

**Objetivo:** examinar a contribuição de operações geométricas com tolerância para o enriquecimento da arquitetura, sem confundir interseção com delimitação semântica.

**Referência científica:** Krijnen et al. (2020), *Validation and inference of geometrical relationships in IFC*, §3.3, DOI [10.46421/2706-6568.37.2020.paper008](https://doi.org/10.46421/2706-6568.37.2020.paper008). [Código dos autores citado no artigo](https://doi.org/10.5281/zenodo.3633653) · [versão arquivada](https://github.com/aothms/IfcOpenShell/tree/v0.7.0-validation).

**Natureza deste notebook:** auditoria executada dos registros reais do ensaio C++/CGAL e integração ao RDF. A compilação e os testes geométricos ocorreram no [notebook de execução Linux/Colab](https://colab.research.google.com/drive/1tJYmSV1nrbo81kncPlphlJDEdX1-laiM); logs, fontes, entradas e saídas estão no pacote verificado. As células abaixo não apresentam a leitura desses registros como nova execução do motor.

**Diferenças para os autores:** IFC interpretado por IfcOpenShell/OpenCascade atual; operações Nef isoladas; preparação por união de componentes é extensão deste estudo, apoiada no [manual CGAL 5.6](https://doc.cgal.org/5.6/Nef_3/index.html). Não executa o programa IfcConvert completo, o achatamento de superfícies ou a inferência de limites arquitetônicos completos.

In [1]:
# Importação das funções de auditoria e RDF. Não executa o motor geométrico.
# RDFLib: https://rdflib.readthedocs.io/en/stable/intro_to_graphs/
from pathlib import Path
import os,json,hashlib,zipfile,importlib.util,sys,subprocess
from urllib.request import urlretrieve
try:
    import rdflib
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','rdflib==7.1.4'])
    import rdflib
RAIZ=Path(os.environ.get('CIV2155_KRIJNEN_RAIZ','CIV2155_Krijnen_Auditoria'))
RAIZ.mkdir(parents=True,exist_ok=True)
print('RDFLib',rdflib.__version__)


RDFLib 7.1.4


## Entradas e integridade

O pacote contém duas variantes: malha composta (v01) e união de componentes fechados (v02). O SHA-256 fixa as entradas e os registros. O IFC original permanece preservado; o ensaio usa coordenadas de projeto em metros e uma seleção de 34 paredes da cozinha. As listas de componentes foram normalizadas para LF, sem alterar os arquivos geométricos.

In [2]:
# Importa o pacote rastreável do experimento e confere seu hash.
# O snapshot fixa os registros, evitando mudanças silenciosas da branch principal.
SNAPSHOT='1f6ec982b6bfcc8d5c6dd5af73bf6da4107cd3c0'
URL='https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/'+SNAPSHOT+'/resultados/krijnen_cgal_v01/Pacote_Reprodutivel_CGAL_v01.zip'
ZIP=RAIZ/'Pacote_Reprodutivel_CGAL_v01.zip'
urlretrieve(URL,ZIP)
assert hashlib.sha256(ZIP.read_bytes()).hexdigest()=='fc0806fcdf0ce0a7a543d00a72275609187bea965b6380d48fb844b14d8016e4'
with zipfile.ZipFile(ZIP) as z:z.extractall(RAIZ)
entrada=json.loads((RAIZ/'execucao_v02/entrada/Registro_Entrada.json').read_text(encoding='utf8'))
print('IFC de origem:',entrada['ifc_original'])
print('SHA256:',entrada['sha256_ifc'])
print('Objetos geométricos:',len(entrada['objetos']))


IFC de origem: TIPO2-ARQ-DTV_R03.ifc
SHA256: 0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a
Objetos geométricos: 35


## Controles e alcance dos resultados

Os controles usam expectativas analíticas de caixas: contato, fresta, sobreposição, contenção, aresta e união de componentes. Não constituem validação arquitetônica independente da creche. O resultado da tentativa com listas CRLF foi corrigido como erro de transporte de arquivos; não é tratado como falha do algoritmo científico.

In [3]:
# Examina saídas efetivamente executadas do C++/CGAL, sem reiniciar o motor.
for variante,esperado in [('execucao_v01',12),('execucao_v02',14)]:
    execucao=json.loads((RAIZ/variante/'Execucao.json').read_text(encoding='utf8'))
    testes=json.loads((RAIZ/variante/'Controles.json').read_text(encoding='utf8'))
    assert execucao['estado']=='concluido' and len(execucao['casos'])==68
    assert testes['aprovados']==testes['total']==esperado
    assert execucao['entradas_off_preservadas']
    estados={s:sum(c['estado']==s for c in execucao['casos']) for s in sorted({c['estado'] for c in execucao['casos']})}
    print(variante,'controles:',esperado,'estados:',estados)


execucao_v01 controles: 12 estados: {'concluido': 28, 'rejeitado_ou_erro': 40}
execucao_v02 controles: 14 estados: {'concluido': 68}


## Incorporação ao grafo do Notebook 1

As operações CGAL seguem a referência de Krijnen; a organização das avaliações é específica deste estudo. A geometria RDF segue o padrão GEOM de Zhang et al. (2018), BimSPARQL, DOI [10.3233/SW-180297](https://doi.org/10.3233/SW-180297).

São acrescentadas avaliações ligadas ao ambiente e à parede, parâmetros, coordenadas e proveniência. As relações IFC2SB anteriores permanecem. Não são promovidas novas afirmações `bot:adjacentElement` ou `bot:intersectingElement` automaticamente. A ausência de `TopAbs_SOLID` em 11 entradas BRep impede interpretar a soma vazia de volumes como zero.

In [4]:
# O módulo verifica a integridade, preserva o RDF anterior e serializa as avaliações.
# Fonte das APIs: RDFLib; critérios de relato próprios do estudo.
spec=importlib.util.spec_from_file_location('civ2155_cgal',RAIZ/'codigo/consolidar.py')
modulo=importlib.util.module_from_spec(spec);spec.loader.exec_module(modulo)
resumo=modulo.consolidate(RAIZ/'execucao_v02',RAIZ/'RDF_Base_v04.ttl',RAIZ/'resultados')
assert resumo['triplas_entrada']==676 and resumo['triplas_saida']==1946
assert resumo['avaliacoes_rdf']==68 and resumo['rdf_entrada_preservado']


{
  "paredes": 34,
  "ensaios": 68,
  "controles": 14,
  "estados_execucao": {
    "concluido": 68
  },
  "interpretacoes": {
    "candidato_por_contato_ou_tolerancia": 11,
    "sem_intersecao_neste_criterio": 3,
    "intersecao_volumetrica_sem_padding": 19,
    "intersecao_3d_nao_quantificada": 1
  },
  "novos_sinais_vs_superficies_piloto": [
    "W07",
    "W13",
    "W14",
    "W15",
    "W16",
    "W17",
    "W18",
    "W19",
    "W20",
    "W21",
    "W22",
    "W23",
    "W24",
    "W25",
    "W26",
    "W27",
    "W28",
    "W29",
    "W30",
    "W31",
    "W32"
  ],
  "intersecoes_sem_padding": 29,
  "intersecoes_com_padding": 31,
  "paredes_com_superficie_coincidente_piloto": 10,
  "volume_ocp_indisponivel_sem_solidos": [
    "W13",
    "W15",
    "W17",
    "W19",
    "W20",
    "W22",
    "W26",
    "W27",
    "W28",
    "W30",
    "W32"
  ],
  "maior_diferenca_volume_nef_ocp_m3": 5.842901273922507e-09,
  "triplas_entrada": 676,
  "triplas_saida": 1946,
  "avaliacoes_rdf": 6

## Consulta e interpretação

O grafo resultante pode ser consultado por parede, parâmetro e estado. As 31 interseções com expansão não equivalem a 31 paredes delimitadoras. O limiar de relato de volume é experimental (1e-8 m³); não é requisito normativo. W22 contém uma interseção tridimensional cuja representação não permitiu quantificação no critério do ensaio. A validação arquitetônica permanece pendente.

In [5]:
# Consulta todas as avaliações no RDF gerado, inclusive casos negativos.
grafo=rdflib.Graph().parse(RAIZ/'resultados/Cozinha_IFC2SB_Com_Avaliacoes_CGAL_v01.ttl',format='turtle')
consulta=(RAIZ/'resultados/Consultar_Avaliacoes_CGAL.rq').read_text(encoding='utf8')
linhas=list(grafo.query(consulta))
assert len(linhas)==68
print('Avaliações consultadas:',len(linhas))
print('Interpretações:',resumo['interpretacoes'])
canonico=RAIZ/'RDF_Consolidado_Publicado.ttl'
urlretrieve('https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/'+SNAPSHOT+'/resultados/krijnen_cgal_v01/Cozinha_IFC2SB_Com_Avaliacoes_CGAL_v01.ttl',canonico)
assert set(grafo)==set(rdflib.Graph().parse(canonico,format='turtle'))
print('Grafo regenerado semanticamente igual ao RDF publicado. A ordem das triplas na serialização pode variar.')
print('IFC original e grafo anterior preservados; completude arquitetônica não demonstrada.')


Avaliações consultadas: 68
Interpretações: {'candidato_por_contato_ou_tolerancia': 11, 'sem_intersecao_neste_criterio': 3, 'intersecao_volumetrica_sem_padding': 19, 'intersecao_3d_nao_quantificada': 1}


Grafo regenerado semanticamente igual ao RDF publicado. A ordem das triplas na serialização pode variar.
IFC original e grafo anterior preservados; completude arquitetônica não demonstrada.


## Reexecução geométrica opcional em Linux

Esta etapa é separada da auditoria. As dependências usadas foram clang 18, CGAL 5.6, GMP 6.3 e MPFR 4.2. No Colab/Linux, a instalação pode ser realizada com `apt-get install clang libcgal-dev libgmp-dev libmpfr-dev`. O comando abaixo usa o supervisor C++ e gera uma réplica separada. Ele não foi acionado nesta auditoria; a execução original está nos logs preservados.

In [6]:
# Opcional: compila e reexecuta o ensaio C++ com o pacote preservado.
# Fonte e diferenças: código/cabeçalho e Proveniencia_Fonte.json no pacote.
REEXECUTAR_MOTOR=False
if REEXECUTAR_MOTOR:
    subprocess.run([sys.executable,str((RAIZ/'codigo/reproduzir_linux.py').resolve())],check=True)
else:
    print('Motor não reiniciado nesta auditoria. Resultados originais preservados.')


Motor não reiniciado nesta auditoria. Resultados originais preservados.
